# 02 — Baseline CNN (known work, reproduced)

Partner A, week 2. Trains the RF-fingerprint CNN classifier. This is not the novelty —
it's the realistic, attackable baseline the rest of the project builds on.

In [ ]:
from src.utils.config import load_config
from src.baseline.train import train_baseline

config = load_config('configs/default.yaml')
seed = config['seed']
split_path = f"data/splits/split_seed{seed}.json"

model = train_baseline(config, split_path, out_dir='results/baseline')

## Evaluate: confusion matrix + per-device accuracy

In [ ]:
import numpy as np
import torch
from torch.utils.data import DataLoader
from sklearn.preprocessing import LabelEncoder

from src.preprocessing.wisig_loader import load_wisig
from src.preprocessing.session_split import load_split, apply_split
from src.baseline.train import IQDataset
from src.baseline.evaluate import predict, per_device_accuracy, build_confusion_matrix

df = load_wisig(config['data']['wisig_root'], subset=config['data']['wisig_subset'], equalized=config['data']['equalized'])
split = load_split(split_path)
parts = apply_split(df, split)
label_encoder = LabelEncoder().fit(df['tx_id'])

test_ds = IQDataset(parts['test'], label_encoder, config['data']['iq_length'])
test_loader = DataLoader(test_ds, batch_size=config['baseline_cnn']['batch_size'], shuffle=False)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
y_true, y_pred, conf = predict(model, test_loader, device)

print('test accuracy:', (y_true == y_pred).mean())
acc_by_device = per_device_accuracy(y_true, y_pred, list(label_encoder.classes_))
print(acc_by_device)

In [ ]:
import matplotlib.pyplot as plt

cm = build_confusion_matrix(y_true, y_pred, len(label_encoder.classes_))
plt.figure(figsize=(8, 8))
plt.imshow(cm, cmap='Blues')
plt.title('Baseline CNN confusion matrix (test)')
plt.xlabel('predicted'); plt.ylabel('true')
plt.colorbar()
plt.savefig('results/baseline/confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()